In [6]:
import requests, os

url = 'https://zenodo.org/records/4435257/files/all_columns_catalog.fits.gz'
dest = 'cf_data/elbadry2021_wide_binaries.fits.gz'

if os.path.exists(dest):
    print(f'Already exists: {dest} ({os.path.getsize(dest)/1e9:.2f} GB)')
else:
    r = requests.get(url, stream=True)
    total = int(r.headers.get('Content-Length', 0))
    downloaded = 0
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024):  # 1 MB chunks
            f.write(chunk)
            downloaded += len(chunk)
            pct = downloaded / total * 100
            print(f'\r{pct:.1f}% ({downloaded/1e9:.2f} / {total/1e9:.2f} GB)', end='', flush=True)
    print(f'\nDone: {dest}')

100.0% (1.42 / 1.42 GB)
Done: cf_data/elbadry2021_wide_binaries.fits.gz


In [9]:
from astropy.io import fits
import numpy as np
import pandas as pd

training = pd.read_csv('cf_data/training_stars.csv')
gaia_ids = set(training['source_id'].dropna().astype(np.int64))
print(f'Training stars to match: {len(gaia_ids)}')

print('Loading El-Badry catalog...')
with fits.open('cf_data/elbadry2021_wide_binaries.fits.gz', memmap=True) as hdul:
    data = hdul[1].data
    sid1 = data['source_id1'].astype(np.int64)
    sid2 = data['source_id2'].astype(np.int64)

    mask = np.isin(sid1, list(gaia_ids)) | np.isin(sid2, list(gaia_ids))
    matched = data[mask]

    # Use np.array(..., dtype=...) to force native (little-endian) byte order
    # FITS is big-endian; pandas merge fails on big-endian buffers on Windows
    eb = pd.DataFrame({
        'source_id1':    np.array(matched['source_id1'],    dtype=np.int64),
        'source_id2':    np.array(matched['source_id2'],    dtype=np.int64),
        'sep_AU':        np.array(matched['sep_AU'],        dtype=np.float64),
        'pairdistance':  np.array(matched['pairdistance'],  dtype=np.float64),
        'R_chance_align':np.array(matched['R_chance_align'],dtype=np.float64),
    })

print(f'Found {len(eb)} matching pairs')
print(eb)


Training stars to match: 53
Loading El-Badry catalog...
Found 10 matching pairs
            source_id1           source_id2        sep_AU  pairdistance  \
0  2477815222028038272  2478001486169801216  14726.224117      0.170021   
1  2485305370114328576  2485211533668928640   1919.777908      0.012346   
2   661793957110898304   661794060190112000   2506.961511      0.009486   
3  1996725077535282944  1996725077535283200    100.942825      0.001708   
4  1600882199829338368  1600882268549215616   1065.181738      0.005154   
5  1379500756257031808  1379500928055726848   1005.126833      0.019444   
6   581100382135253760   581103581886377728   4835.132779      0.044292   
7  2005798350576886144  2005804153064109056   1686.714214      0.021361   
8  2826254717479033344  2826254713186397440    373.799898      0.002638   
9  3175193288128400768  3175193288128400896    604.843871      0.003588   

   R_chance_align  
0    1.233319e-02  
1    2.865998e-05  
2    3.393964e-05  
3    5.604461e

In [10]:

# Identify which component is the training star and which is the companion
both_match = eb['source_id1'].isin(gaia_ids) & eb['source_id2'].isin(gaia_ids)
print(f'Pairs where BOTH components are training stars: {both_match.sum()}')

eb['training_star_id'] = np.where(eb['source_id1'].isin(gaia_ids), eb['source_id1'], eb['source_id2'])
eb['companion_id']     = np.where(eb['source_id1'].isin(gaia_ids), eb['source_id2'], eb['source_id1'])

# Merge in training star name + source_paper
id_to_name = training.set_index('source_id')[['star_name', 'source_paper', 'age_gyr', 'age_method', 'prot_days']].drop_duplicates()
eb = eb.merge(id_to_name, left_on='training_star_id', right_index=True, how='left')

# Flag which came from Pass2022 (already had El-Badry data) vs new matches
eb['already_in_pass'] = eb['source_paper'] == 'Pass2022'

print(f"\nNew (Engle) matches: {(~eb['already_in_pass']).sum()}")
print(f"Pass2022 matches (already known): {eb['already_in_pass'].sum()}")
print()
print(eb[['star_name', 'source_paper', 'age_method', 'age_gyr', 'prot_days',
          'companion_id', 'sep_AU', 'R_chance_align']].to_string())


Pairs where BOTH components are training stars: 0

New (Engle) matches: 6
Pass2022 matches (already known): 5

                 star_name source_paper     age_method   age_gyr  prot_days         companion_id        sep_AU  R_chance_align
0                G 271-110     Pass2022  fgk_companion  0.270778      1.060  2477815222028038272  14726.224117    1.233319e-02
1                LP 587-54    Engle2023   wd_companion  6.340000     59.500  2485211533668928640   1919.777908    2.865998e-05
2              HIP 43232 B    Engle2023   ms_companion  3.950000     41.300   661793957110898304   2506.961511    3.393964e-05
3  2MASS J23095781+5506472     Pass2022   wd_companion       NaN    104.100  1996725077535283200    100.942825    5.604461e-08
3  2MASS J23095781+5506472    Engle2023   wd_companion  5.470000    104.100  1996725077535283200    100.942825    5.604461e-08
4                  G201-40    Engle2023   wd_companion  5.160000     77.600  1600882268549215616   1065.181738    1.088657e-10


In [ ]:
# Drop duplicate rows where the same training star matched twice
# (e.g. 2MASS J23095781+5506472 appears in both Pass2022 and Engle2023)
# Keep the row with a non-null age_gyr where possible
eb_out = (eb.sort_values('age_gyr', na_position='last')
            .drop_duplicates(subset='training_star_id', keep='first')
            .reset_index(drop=True))

print(f'Pairs after dedup: {len(eb_out)}')
eb_out[['star_name', 'source_paper', 'age_method', 'age_gyr', 'prot_days',
        'training_star_id', 'companion_id', 'sep_AU', 'R_chance_align']].to_csv(
    'cf_data/elbadry_matches.csv', index=False)
print('Saved to cf_data/elbadry_matches.csv')


Pairs after dedup: 10
Saved to cf_data/elbadry_matches.csv


: 